In [23]:
!pip install nba_api
import nba_api # Added import nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics, leaguegamefinder
import time
from io import StringIO
from datetime import datetime
import pytz
import json
from scipy.stats import zscore
from nba_api.stats.library.parameters import SeasonTypeAllStar # Import SeasonTypeAllStar

# print(f"nba_api version: {nba_api.__version__}") # Print nba_api version - Removed version check

pd.set_option('display.max_columns', None)

# Team name mappings
team_name_mapping = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS",
}
team_abbreviation_to_name = {abbr: full_name for full_name, abbr in team_name_mapping.items()}

# Re-defined fetch_schedule to return game IDs as well
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'game_id': game['gid'], # Include game ID
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

def fetch_team_metrics():
    """Fetch current team efficiency metrics."""
    team_metrics_data = teamestimatedmetrics.TeamEstimatedMetrics(season="2024-25", season_type="Regular Season")
    team_metrics = team_metrics_data.get_data_frames()[0]
    team_metrics = team_metrics[["TEAM_NAME", "TEAM_ID", "E_OFF_RATING", "E_DEF_RATING", "E_PACE"]]
    team_metrics["TEAM_ABBREVIATION"] = team_metrics["TEAM_NAME"].map(team_name_mapping)
    return team_metrics

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", # Removed SeasonType parameter
            per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Fetch player game logs for calculating historical averages
def fetch_player_game_logs(season="2023-24", season_type="Regular Season"): # Using a recent season for example
    all_player_game_logs = []
    player_list = leaguedashplayerstats.LeagueDashPlayerStats(season=season).get_data_frames()[0] # Removed SeasonType parameter
    player_ids = player_list['PLAYER_ID'].tolist()

    for player_id in player_ids:
        try:
            game_finder = leaguegamefinder.LeagueGameFinder(
                player_id_nullable=player_id,
                season_nullable=season,
                season_type_nullable=season_type
            )
            games = game_finder.get_data_frames()[0]
            if not games.empty:
                player_games = games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'PTS', 'REB', 'AST', 'TEAM_ID']].copy() # Use .copy() to avoid chained indexing
                player_games.loc[:, 'PLAYER_ID'] = player_id # Corrected SettingWithCopyWarning using .loc and assign to copy
                all_player_game_logs.append(player_games)
            time.sleep(0.3) # Rate limiting
        except Exception as e:
            print(f"Error fetching game logs for player ID {player_id}: {e}")

    return pd.concat(all_player_game_logs) if all_player_game_logs else pd.DataFrame()


# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Fetch NBA injury data and store it in a dictionary
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    # Dictionary to store player injury statuses
    players_injuries = {}

    for team in data.get('injuries', []):
        for injury in team.get('injuries', []):
            player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
            status = injury.get('status', '').strip()  # Get status and remove whitespace

            # Handle missing or invalid statuses
            if not status or status.lower() == 'nan':  # Check for empty or "nan" values
                status = 'Healthy'

            # Store the player's status, avoiding duplicates
            players_injuries[player_name] = status
else:
    print("Failed to retrieve injury data.")
    players_injuries = {}  # Initialize as empty dictionary if data retrieval fails


# Create the fetch_injuries function to return the injury data
def fetch_injuries():
    """Returns a dictionary containing player injury statuses."""
    return players_injuries

# Fetch player positions, team defense, player stats, etc. (keep these functions as is)

# Merge the injury status with player data and ensure no duplicates
def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df

# Project PRA for each player against opposing teams (REVISED for PRA projection)
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats for current season...") # Use current season stats as a base
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()
    time.sleep(2)

    print("Fetching injuries...")
    injuries = fetch_injuries()
    time.sleep(2)

    print("Fetching player game logs for historical averages...") # Fetch game logs
    player_game_logs_df = fetch_player_game_logs()
    time.sleep(2)

    if player_stats_df is None or team_defense_df is None or player_positions_df is None or player_game_logs_df is None:
        print("Error: One or more dataframes are empty. Returning empty DataFrame.")
        return pd.DataFrame()

    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Calculate historical average PRA from game logs
    print("Calculating historical average PRA...")
    player_game_logs_df['PRA'] = player_game_logs_df['PTS'] + player_game_logs_df['REB'] + player_game_logs_df['AST']
    avg_pra = player_game_logs_df.groupby('PLAYER_ID')['PRA'].mean().to_dict()
    dvp_df['Avg_PRA'] = dvp_df['PLAYER_ID'].map(avg_pra).fillna(30) # Fill NaN with a reasonable default (e.g., 30 PRA - adjust as needed)


    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST", 'Avg_PRA'] # Include Avg_PRA
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    print("Projecting PRA based on DVP and historical averages...")
    # Projection now primarily based on historical average PRA and DVP adjustment
    dvp_df["Proj_PRA"] = (dvp_df["Avg_PRA"] *
                         ((dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean()) +  # DVP Points factor
                          (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean()) +  # DVP Rebounds factor
                          (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())) / 3).round(1) # DVP Assists factor, averaged

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PRA', 'E_USG_PCT']].copy() # Only keep Proj_PRA
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    proj_pra_df = merge_with_injuries(proj_pra_df, injuries)

    print("PRA Projection completed successfully!")
    return proj_pra_df if not proj_pra_df.empty else pd.DataFrame()


def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df['Injury_Status'] = proj_pra_df['Injury_Status'].fillna('Healthy')
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df


# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz", "WAS": "Washington Wizards"
}

    for game in todays_games:
        game_id = game['game_id'] # Get game ID
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame ID: {game_id} - {home_team_full} vs {away_team_full}") # Include Game ID in output
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats (now showing Proj_PRA)
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | Pos: {row['Pos']:<3} | Proj_PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | Pos: {row['Pos']:<3} | Proj_PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

# Run the pipeline
if __name__ == "__main__":
    run_pipeline_for_todays_games()

Fetching today's schedule...
Projecting PRA for players...
Fetching player positions...
Fetching team defensive stats...
Fetching player stats for current season...
Fetching player usage percentage...
Fetching injuries...
Fetching player game logs for historical averages...
Error fetching game logs for player ID 203076: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error fetching game logs for player ID 1630162: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)


KeyboardInterrupt: 

In [26]:
!pip install nba_api
import nba_api # Added import nba_api
import pandas as pd
import requests
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats, playerestimatedmetrics, leaguegamefinder
import time
from io import StringIO
from datetime import datetime
import pytz
import json
from scipy.stats import zscore
from nba_api.stats.library.parameters import SeasonTypeAllStar # Import SeasonTypeAllStar

# print(f"nba_api version: {nba_api.__version__}") # Print nba_api version - Removed version check

pd.set_option('display.max_columns', None)

# Team name mappings
team_name_mapping = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS",
}
team_abbreviation_to_name = {abbr: full_name for full_name, abbr in team_name_mapping.items()}

# Re-defined fetch_schedule to return game IDs as well
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'game_id': game['gid'], # Include game ID
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

def fetch_team_metrics():
    """Fetch current team efficiency metrics."""
    team_metrics_data = teamestimatedmetrics.TeamEstimatedMetrics(season="2024-25", season_type="Regular Season")
    team_metrics = team_metrics_data.get_data_frames()[0]
    team_metrics = team_metrics[["TEAM_NAME", "TEAM_ID", "E_OFF_RATING", "E_DEF_RATING", "E_PACE"]]
    team_metrics["TEAM_ABBREVIATION"] = team_metrics["TEAM_NAME"].map(team_name_mapping)
    return team_metrics

# Step 1: Scrape player positions from Basketball-Reference
def get_player_positions():
    url = "https://www.basketball-reference.com/leagues/NBA_2024_per_game.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # Extract player stats table
    table = soup.find("table", {"id": "per_game_stats"})
    if table is None:
        raise ValueError("Table not found on Basketball-Reference. The structure may have changed.")

    df = pd.read_html(StringIO(str(table)))[0]

    # Ensure we use the correct column name for team
    if 'Team' in df.columns:
        df.rename(columns={'Team': 'TEAM_ABBREVIATION'}, inplace=True)
    else:
        raise KeyError("Column 'Team' not found. Check Basketball-Reference structure.")

    # Remove duplicated headers (Basketball-Reference has repeated "Player" rows)
    df = df[df['Player'] != 'Player']

    return df[['Player', 'Pos', 'TEAM_ABBREVIATION']]

# Step 2: Fetch team defensive stats from NBA API
def get_team_defense_stats():
    team_defense = leaguedashteamstats.LeagueDashTeamStats(
        measure_type_detailed_defense="Opponent",
        per_mode_detailed="PerGame",
        season="2024-25"
    ).get_data_frames()[0]

    # Keep only relevant columns
    team_defense = team_defense[['TEAM_ID', 'TEAM_NAME', 'OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']]
    return team_defense

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        player_stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        # Select necessary columns
        usg_data = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Step 3: Fetch player stats (use LeaguedashPlayerStats for current season data)
def fetch_player_stats():
    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season="2024-25", # Removed SeasonType parameter
            per_mode_detailed="PerGame"
        ).get_data_frames()[0]

        player_stats = player_stats[['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PTS', 'REB', 'AST']]
        return player_stats
    except Exception as e:
        print(f"Error fetching player stats: {e}")
        return pd.DataFrame()

# Fetch player game logs for calculating historical averages with retry logic
def fetch_player_game_logs(season="2023-24", season_type="Regular Season", timeout=60, max_retries=3, retry_delay=5): # Added retry parameters
    all_player_game_logs = []
    player_list = leaguedashplayerstats.LeagueDashPlayerStats(season=season).get_data_frames()[0] # Removed SeasonType parameter
    player_ids = player_list['PLAYER_ID'].tolist()

    for player_id in player_ids:
        for retry_attempt in range(max_retries): # Retry loop
            try:
                game_finder = leaguegamefinder.LeagueGameFinder(
                    player_id_nullable=player_id,
                    season_nullable=season,
                    season_type_nullable=season_type,
                    timeout=timeout # Pass timeout parameter here
                )
                games = game_finder.get_data_frames()[0]
                if not games.empty:
                    player_games = games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'PTS', 'REB', 'AST', 'TEAM_ID']].copy() # Use .copy() to avoid chained indexing
                    player_games.loc[:, 'PLAYER_ID'] = player_id # Corrected SettingWithCopyWarning using .loc and assign to copy
                    all_player_game_logs.append(player_games)
                    break # Success, break out of retry loop
                else:
                    break # No games found, no need to retry, just continue to next player

            except Exception as e:
                print(f"Error fetching game logs for player ID {player_id}, attempt {retry_attempt + 1}/{max_retries}: {e}")
                if retry_attempt < max_retries - 1:
                    time.sleep(retry_delay * (retry_attempt + 1)) # Exponential backoff
                else:
                    print(f"Max retries reached for player ID {player_id}. Skipping.") # Give up after max retries
        else: # else block associated with for loop, executes if no 'break' was hit in the loop
            continue # Continue to the next player even if retries failed, to process other players

        time.sleep(1) # Increased rate limiting to 1 second per player

    return pd.concat(all_player_game_logs) if all_player_game_logs else pd.DataFrame()


# Step 5: Merge data & calculate DVP with E_USG_PCT
def calculate_dvp():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats...")
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    # Merge player positions with player stats
    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    # Merge with team defensive stats using TEAM_ID
    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    # Convert numeric columns before aggregation, including 'E_USG_PCT'
    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST"]
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    # Calculate DVP (Defense vs Position)
    dvp_summary = dvp_df.groupby("Pos")[['OPP_PTS', 'OPP_FG3M', 'OPP_REB', 'OPP_AST']].mean().reset_index()

    return dvp_summary

# Fetch NBA injury data and store it in a dictionary
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    # Dictionary to store player injury statuses
    players_injuries = {}

    for team in data.get('injuries', []):
        for injury in team.get('injuries', []):
            player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
            status = injury.get('status', '').strip()  # Get status and remove whitespace

            # Handle missing or invalid statuses
            if not status or status.lower() == 'nan':  # Check for empty or "nan" values
                status = 'Healthy'

            # Store the player's status, avoiding duplicates
            players_injuries[player_name] = status
else:
    print("Failed to retrieve injury data.")
    players_injuries = {}  # Initialize as empty dictionary if data retrieval fails


# Create the fetch_injuries function to return the injury data
def fetch_injuries():
    """Returns a dictionary containing player injury statuses."""
    return players_injuries

# Fetch player positions, team defense, player stats, etc. (keep these functions as is)

# Merge the injury status with player data and ensure no duplicates
def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df

# Project PRA for each player against opposing teams (REVISED for PRA projection)
def project_pra():
    print("Fetching player positions...")
    player_positions_df = get_player_positions()
    time.sleep(2)

    print("Fetching team defensive stats...")
    team_defense_df = get_team_defense_stats()
    time.sleep(2)

    print("Fetching player stats for current season...") # Use current season stats as a base
    player_stats_df = fetch_player_stats()
    time.sleep(2)

    print("Fetching player usage percentage...")
    usg_data = fetch_player_usg()
    time.sleep(2)

    print("Fetching injuries...")
    injuries = fetch_injuries()
    time.sleep(2)

    print("Fetching player game logs for historical averages...") # Fetch game logs
    player_game_logs_df = fetch_player_game_logs()
    time.sleep(2)

    if player_stats_df is None or team_defense_df is None or player_positions_df is None or player_game_logs_df is None:
        print("Error: One or more dataframes are empty. Returning empty DataFrame.")
        return pd.DataFrame()

    print("Merging player stats with positions...")
    merged_df = player_stats_df.merge(player_positions_df, left_on="PLAYER_NAME", right_on="Player", how="left")
    merged_df.drop(columns=["Player"], inplace=True)

    print("Merging with team defensive stats...")
    dvp_df = merged_df.merge(team_defense_df, on="TEAM_ID", how="left")

    dvp_df['E_USG_PCT'] = dvp_df['PLAYER_ID'].map(lambda x: usg_data.get(x, {}).get('E_USG_PCT', None))

    # Calculate historical average PRA from game logs
    print("Calculating historical average PRA...")
    player_game_logs_df['PRA'] = player_game_logs_df['PTS'] + player_game_logs_df['REB'] + player_game_logs_df['AST']
    avg_pra = player_game_logs_df.groupby('PLAYER_ID')['PRA'].mean().to_dict()
    dvp_df['Avg_PRA'] = dvp_df['PLAYER_ID'].map(avg_pra).fillna(30) # Fill NaN with a reasonable default (e.g., 30 PRA - adjust as needed)


    numeric_cols = ["PTS", "REB", "AST", "OPP_PTS", "OPP_FG3M", "OPP_REB", "OPP_AST", 'Avg_PRA'] # Include Avg_PRA
    for col in numeric_cols:
        dvp_df[col] = pd.to_numeric(dvp_df[col], errors="coerce")

    print("Projecting PRA based on DVP and historical averages...")
    # Projection now primarily based on historical average PRA and DVP adjustment
    dvp_df["Proj_PRA"] = (dvp_df["Avg_PRA"] *
                         ((dvp_df["OPP_PTS"] / dvp_df["OPP_PTS"].mean()) +  # DVP Points factor
                          (dvp_df["OPP_REB"] / dvp_df["OPP_REB"].mean()) +  # DVP Rebounds factor
                          (dvp_df["OPP_AST"] / dvp_df["OPP_AST"].mean())) / 3).round(1) # DVP Assists factor, averaged

    proj_pra_df = dvp_df[['PLAYER_NAME', 'TEAM_ID', 'Pos', 'Proj_PRA', 'E_USG_PCT']].copy() # Only keep Proj_PRA
    proj_pra_df = proj_pra_df.sort_values(by='E_USG_PCT', ascending=False)

    proj_pra_df = merge_with_injuries(proj_pra_df, injuries)

    print("PRA Projection completed successfully!")
    return proj_pra_df if not proj_pra_df.empty else pd.DataFrame()


def merge_with_injuries(proj_pra_df, injuries):
    proj_pra_df['Injury_Status'] = proj_pra_df['PLAYER_NAME'].map(injuries)
    proj_pra_df['Injury_Status'] = proj_pra_df['Injury_Status'].fillna('Healthy')
    proj_pra_df.drop_duplicates(subset=['PLAYER_NAME'], inplace=True)
    return proj_pra_df


# Step 7: Run the full pipeline for today's games
def run_pipeline_for_todays_games():
    print("Fetching today's schedule...")
    todays_games = fetch_schedule()

    if not todays_games:
        print("No games for today.")
        return

    print("Projecting PRA for players...")
    projected_pra = project_pra()

    # Full team names (you can expand this dictionary as needed)
    team_names = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz", "WAS": "Washington Wizards"
}

    for game in todays_games:
        game_id = game['game_id'] # Get game ID
        home_team_id = game['home_team_id']
        away_team_id = game['away_team_id']

        # Get home and away team player projections
        home_team_players = projected_pra[projected_pra['TEAM_ID'] == home_team_id]
        away_team_players = projected_pra[projected_pra['TEAM_ID'] == away_team_id]

        # Map abbreviations to full team names
        home_team_full = team_names.get(game['home_team'], game['home_team'])
        away_team_full = team_names.get(game['away_team'], game['away_team'])

        # Print projections for today's game
        print(f"\nGame ID: {game_id} - {home_team_full} vs {away_team_full}") # Include Game ID in output
        print(f"\nProjected PRA for Home Team ({home_team_full}):")

        # Format player stats (now showing Proj_PRA)
        for _, row in home_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | Pos: {row['Pos']:<3} | Proj_PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

        print(f"\nProjected PRA for Away Team ({away_team_full}):")
        for _, row in away_team_players.iterrows():
            print(f"{row['PLAYER_NAME']:<25} | Pos: {row['Pos']:<3} | Proj_PRA: {row['Proj_PRA']:<5.1f} | USG: {row['E_USG_PCT']:<5.2f} | Injury: {row['Injury_Status']}")

# Run the pipeline
if __name__ == "__main__":
    run_pipeline_for_todays_games()

Fetching today's schedule...
Projecting PRA for players...
Fetching player positions...
Fetching team defensive stats...


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)